In [2]:
#Import library yang diperlukan
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os

# Load semua file
df = pd.read_csv('../data/raw/transactions_raw.csv')
products = pd.read_csv('../data/raw/products.csv')
customers = pd.read_csv('../data/raw/customers.csv')
stores = pd.read_csv('../data/raw/stores.csv')

# Tampilkan informasi dasar tentang dataset
print(f"Shape: {df.shape}")
print(f"\nKolom: {df.columns.tolist()}")
df.head()

Shape: (5610, 15)

Kolom: ['order_id', 'order_date', 'customer_id', 'store_id', 'product_id', 'product_name', 'category', 'qty', 'unit_price', 'discount_pct', 'discount_amount', 'total_price', 'payment_method', 'order_status', 'city']


,order_id,order_date,customer_id,store_id,product_id,product_name,category,qty,unit_price,discount_pct,discount_amount,total_price,payment_method,order_status,city
0,ORD01021,2024-08-13,CST0008,STR028,PRD0043,Tas Sekolah Anak SD,Perlengkapan Bayi & Anak,1,103400.0,15.0,15500.0,87900.0,QRIS,delivered,Palembang
1,ORD04039,2023-10-08,CST0147,STR035,PRD0006,Jaket Bomber,Fashion & Pakaian,1,321800.0,0.0,0.0,321800.0,LinkAja,cancelled,Surabaya
2,ORD03317,2024-11-07,CST0744,STR016,PRD0054,Lukisan Kanvas UMKM,Kerajinan & Souvenir,2,2068000.0,0.0,0.0,413600.0,GoPay,cancelled,Medan
3,ORD02869,2023-02-08,CST0234,STR031,PRD0004,Celana Chino Pria,Fashion & Pakaian,2,164500.0,15.0,49400.0,279700.0,COD,delivered,Palembang
4,ORD03448,2023-06-26,CST0782,STR042,PRD0052,Gelang Batok Kelapa,Kerajinan & Souvenir,1,24900.0,0.0,0.0,24900.0,GoPay,cancelled,Bogor


In [3]:
# Overview kondisi data
print("=" * 50)
print("1. INFO TIPE DATA")
print("=" * 50)
df.info()

print("\n" + "=" * 50)
print("2. MISSING VALUES")
print("=" * 50)
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)
missing_report = pd.DataFrame({
    'missing_count': missing,
    'missing_pct': missing_pct
}).query('missing_count > 0')
print(missing_report)

print("\n" + "=" * 50)
print("3. DUPLICATE ROWS")
print("=" * 50)
print(f"Jumlah duplikat: {df.duplicated().sum()}")

print("\n" + "=" * 50)
print("4. STATISTIK NUMERIK")
print("=" * 50)
print(df[['qty', 'unit_price', 'discount_pct', 'total_price']].describe())

print("\n" + "=" * 50)
print("5. UNIQUE VALUES - KOLOM KATEGORIK")
print("=" * 50)
for col in ['category', 'payment_method', 'order_status']:
    print(f"\n[{col}] — {df[col].nunique()} unique:")
    print(df[col].value_counts())

1. INFO TIPE DATA
<class 'pandas.DataFrame'>
RangeIndex: 5610 entries, 0 to 5609
Data columns (total 15 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   order_id         5610 non-null   str    
 1   order_date       5610 non-null   str    
 2   customer_id      5610 non-null   str    
 3   store_id         5610 non-null   str    
 4   product_id       5610 non-null   str    
 5   product_name     5610 non-null   str    
 6   category         5610 non-null   str    
 7   qty              5610 non-null   int64  
 8   unit_price       5610 non-null   float64
 9   discount_pct     5440 non-null   float64
 10  discount_amount  5610 non-null   float64
 11  total_price      5610 non-null   float64
 12  payment_method   5333 non-null   str    
 13  order_status     5610 non-null   str    
 14  city             5495 non-null   str    
dtypes: float64(4), int64(1), str(10)
memory usage: 1.2 MB

2. MISSING VALUES
                missing_co

In [4]:
# Cek duplikat berdasarkan order_id karena order_id seharusnya unik untuk setiap transaksi
print(f"Duplikat seluruh baris: {df.duplicated().sum()}")
print(f"Duplikat order_id: {df.duplicated(subset='order_id').sum()}")

# Drop duplikat — keep first occurrence
df = df.drop_duplicates(subset='order_id', keep='first').reset_index(drop=True)
print(f"\nShape setelah drop duplikat: {df.shape}")

Duplikat seluruh baris: 97
Duplikat order_id: 110

Shape setelah drop duplikat: (5500, 15)


In [5]:
# Lihat sampel format yang rusak
print("Sampel format tanggal tidak konsisten:")
date_samples = df[df['order_date'].str.contains('/', na=False)]['order_date'].head(10)
print(date_samples.tolist())

# Parse semua format sekaligus dengan dayfirst=True
df['order_date'] = pd.to_datetime(df['order_date'], dayfirst=True, errors='coerce')

# Cek hasil
nat_count = df['order_date'].isna().sum()
print(f"\nGagal diparse (NaT): {nat_count} rows")
print(f"Range tanggal: {df['order_date'].min()} → {df['order_date'].max()}")

# Ekstrak fitur waktu 
df['year'] = df['order_date'].dt.year
df['month'] = df['order_date'].dt.month
df['quarter'] = df['order_date'].dt.quarter
df['month_name'] = df['order_date'].dt.strftime('%b')

Sampel format tanggal tidak konsisten:
['2024/08/06', '07/07/2024', '2024/09/14', '2023/12/05', '29/04/2024', '2023/05/08', '02/06/2024', '07/10/2023', '2023/08/29', '06/01/2024']

Gagal diparse (NaT): 166 rows
Range tanggal: 2023-01-01 00:00:00 → 2024-12-31 00:00:00


C:\Users\ssput\AppData\Local\Temp\ipykernel_6476\914640214.py:7: UserWarning: Parsing dates in %Y-%m-%d format when dayfirst=True was specified. Pass `dayfirst=False` or specify a format to silence this warning.
  df['order_date'] = pd.to_datetime(df['order_date'], dayfirst=True, errors='coerce')


In [6]:
# Standardisasi category → title case + strip spasi
print("Sebelum:", df['category'].unique())

df['category'] = (df['category']
                  .str.strip()
                  .str.title()
                  .str.replace('&amp;', '&')
                 )

# Manual mapping untuk kategori yang typo/disingkat
category_mapping = {
    'Fashion&Pakaian'          : 'Fashion & Pakaian',
    'Makanan&Minuman'          : 'Makanan & Minuman',
    'Makanan'                  : 'Makanan & Minuman',
    'Kecantikan&Perawatan'     : 'Kecantikan & Perawatan',
    'Kecantikan'               : 'Kecantikan & Perawatan',
}
df['category'] = df['category'].replace(category_mapping)
print("Sesudah:", df['category'].unique())

# Standardisasi payment_method
df['payment_method'] = df['payment_method'].str.strip().str.title()
print("\nPayment methods:", sorted(df['payment_method'].dropna().unique()))

Sebelum: <ArrowStringArray>
['Perlengkapan Bayi & Anak',        'Fashion & Pakaian',
     'Kerajinan & Souvenir',       'Olahraga & Outdoor',
   'Kecantikan & Perawatan',    'Elektronik & Aksesori',
        'Makanan & Minuman',             'Rumah Tangga',
     'Kecantikan&Perawatan',        'makanan & minuman',
        'FASHION & PAKAIAN']
Length: 11, dtype: str
Sesudah: <ArrowStringArray>
['Perlengkapan Bayi & Anak',        'Fashion & Pakaian',
     'Kerajinan & Souvenir',       'Olahraga & Outdoor',
   'Kecantikan & Perawatan',    'Elektronik & Aksesori',
        'Makanan & Minuman',             'Rumah Tangga']
Length: 8, dtype: str

Payment methods: ['Alfamart', 'Cod', 'Dana', 'Gopay', 'Indomaret', 'Linkaja', 'Ovo', 'Qris', 'Shopeepay', 'Transfer Bca', 'Transfer Bri', 'Transfer Mandiri']


In [7]:
print("Missing sebelum:")
print(df[['payment_method', 'discount_pct', 'city']].isnull().sum())

# payment_method → isi dengan 'Unknown' (tidak tahu metode = bukan 0)
df['payment_method'] = df['payment_method'].fillna('Unknown')

# discount_pct → isi dengan 0 (asumsi bisnis: tidak ada diskon jika tidak tercatat)
df['discount_pct'] = df['discount_pct'].fillna(0)

# city → isi dari store_id via lookup ke stores.csv
city_lookup = stores.set_index('store_id')['city']
df['city'] = df.apply(
    lambda row: city_lookup.get(row['store_id'], row['city'])
    if pd.isna(row['city']) else row['city'],
    axis=1
)

print("\nMissing sesudah:")
print(df[['payment_method', 'discount_pct', 'city']].isnull().sum())

Missing sebelum:
payment_method    275
discount_pct      165
city              110
dtype: int64

Missing sesudah:
payment_method    0
discount_pct      0
city              0
dtype: int64


In [8]:
# --- Negative qty ---
print(f"Qty negatif: {(df['qty'] < 0).sum()} rows")

df.loc[(df['qty'] < 0) & (df['order_status'] == 'returned'), 'qty'] = \
    df.loc[(df['qty'] < 0) & (df['order_status'] == 'returned'), 'qty'].abs()
df = df[df['qty'] > 0].reset_index(drop=True)

# --- Outlier unit_price — FIXED: tanpa groupby apply ---
before = len(df)

Q1 = df.groupby('category')['unit_price'].transform('quantile', 0.25)
Q3 = df.groupby('category')['unit_price'].transform('quantile', 0.75)
IQR = Q3 - Q1
upper_bound = Q3 + 3 * IQR

df = df[df['unit_price'] <= upper_bound].reset_index(drop=True)
print(f"Outlier harga dihapus: {before - len(df)} rows")
print(f"Shape setelah: {df.shape}")

Qty negatif: 28 rows
Outlier harga dihapus: 64 rows
Shape setelah: (5414, 19)


In [9]:
# Rekalkukasi total_price karena ada baris yang unit_price-nya sudah diubah
df['total_price'] = df['unit_price'] * df['qty'] * (1 - df['discount_pct'] / 100)
df['total_price'] = df['total_price'].round(0)

# Validasi: total_price tidak boleh negatif
assert (df['total_price'] >= 0).all(), "Ada total_price negatif!"

# Final report
print("=" * 50)
print("CLEANING SUMMARY")
print("=" * 50)
print(f"Shape akhir    : {df.shape}")
print(f"Missing values : {df.isnull().sum().sum()}")
print(f"Duplikat       : {df.duplicated(subset='order_id').sum()}")
print(f"Date range     : {df['order_date'].min().date()} → {df['order_date'].max().date()}")
print(f"Kategori unik  : {df['category'].nunique()}")
print(f"Payment method : {df['payment_method'].nunique()}")

CLEANING SUMMARY
Shape akhir    : (5414, 19)
Missing values : 825
Duplikat       : 0
Date range     : 2023-01-01 → 2024-12-31
Kategori unik  : 8
Payment method : 13


In [10]:
os.makedirs('../data/processed', exist_ok=True)
df.to_csv('../data/processed/transactions_clean.csv', index=False)
print("✓ Disimpan ke: data/processed/transactions_clean.csv")

✓ Disimpan ke: data/processed/transactions_clean.csv
